# Notebook 1: OpenAI Agents SDK — Basics

**What you'll learn:** What is the Agents SDK, why it exists, its core primitives, and your first agent in 5 lines.

---

## What is an AI Agent?

An **AI Agent** is NOT just a chatbot. It is:

```
Agent = LLM + Instructions + Tools + Decision-Making Loop
```

A chatbot answers questions. An agent **takes actions**, **makes decisions**, and **completes tasks autonomously**.

### Real-World Analogy
- **Chatbot** = A librarian who answers your questions
- **Agent** = A personal assistant who books flights, sends emails, checks weather, and plans your day

---

## Why OpenAI Agents SDK?

| Feature | Raw LLM API | LangChain | **Agents SDK** |
|---|---|---|---|
| Complexity | Manual everything | Heavy abstractions | **Minimal primitives** |
| Learning Curve | Low (but manual) | Steep | **Gentle** |
| Production Ready | DIY | Yes | **Yes** |
| Tool Calling | Manual JSON | Chain-based | **Decorator-based** |
| Multi-Agent | Build yourself | Complex graphs | **Built-in handoffs** |
| Provider Lock-in | OpenAI only | Any | **Any (100+ LLMs)** |

The SDK evolved from OpenAI's experimental **Swarm** project into a production-grade framework.

---

## The 4 Core Primitives

The entire SDK is built on just **4 concepts**:

```
1. Agent        → The brain (LLM + instructions + tools)
2. Runner       → The executor (runs the agent loop)
3. Tools        → The hands (functions the agent can call)
4. Handoffs     → The delegation (agent-to-agent transfer)
```

That's it. No graphs, no nodes, no edges, no state machines.

## Installation

In [ ]:
# Install the SDK
# uv add openai-agents

## Setup API Key

By default, the SDK uses OpenAI. Set your key:

In [2]:
import os

# Option 1: Set directly (for learning only — never commit real keys!)
# os.environ["OPENAI_API_KEY"] = "sk-your-key-here"

# Option 2: Use .env file (recommended)
# uv add python-dotenv
# from dotenv import load_dotenv
# load_dotenv()

---

## Your First Agent — Hello World

In [ ]:
from agents import Agent, Runner

# Step 1: Create an Agent
agent = Agent(
    name="Greeter",                                    # Give it a name
    instructions="You are a friendly assistant. Be concise.",  # Its personality/role
)

# # Step 2: Run it (sync version — good for scripts)
# result = Runner.run_sync(agent, "Hello! What can you do?")

# # Step 3: Get the output
# print(result.final_output)

### Breaking it down:

```python
Agent(name, instructions)   # WHO the agent is
Runner.run_sync(agent, input) # RUN the agent with user input  
result.final_output           # GET what the agent produced
```

**That's the entire mental model.** Everything else builds on this.

---

## Async Version (For Jupyter Notebooks)

`Runner.run_sync()` doesn't work inside Jupyter because Jupyter already has an event loop running.  
Use `await Runner.run()` instead:

In [ ]:
from agents import Agent, Runner

agent = Agent(
    name="Greeter",
    instructions="You are a friendly assistant. Be concise.",
    # model="gpt-5.4-mini"
)

# In Jupyter, use await directly (no asyncio.run needed)
result = await Runner.run(agent, "What is the capital of Pakistan?")
print(result.final_output)

---

## Understanding the Result Object

The `RunResult` object contains much more than just the output:

In [ ]:
from agents import Agent, Runner

agent = Agent(name="Analyst", instructions="You analyze topics briefly.")

result = await Runner.run(agent, "Why is Python popular for AI?")

# The key properties of RunResult:
print("=" * 50)
print(f"Final Output:  {result.final_output[:100]}...")   # The text response
print(f"Last Agent:    {result.last_agent.name}")          # Which agent finished
print(f"New Items:     {len(result.new_items)}")           # Items generated during run
print(f"Raw Responses: {len(result.raw_responses)}")       # Raw LLM responses
print("=" * 50)

---

## The Agent Loop — How It Actually Works

When you call `Runner.run()`, here's what happens internally:

```
┌─────────────────────────────────────────────┐
│           Runner.run(agent, input)          │
│                                             │
│  ┌──────────────────────────────────────┐   │
│  │ 1. Send input + instructions to LLM  │   │
│  │ 2. LLM responds                      │   │
│  │ 3. Check response:                   │   │
│  │    ├─ Final output? → RETURN result  │   │
│  │    ├─ Tool call?    → Execute tool,  │   │
│  │    │                  loop back to 1 │   │
│  │    └─ Handoff?      → Switch agent,  │   │
│  │                       loop back to 1 │   │
│  └──────────────────────────────────────┘   │
│                                             │
│  This loop IS the "agentic" behavior.       │
│  The LLM DECIDES what to do next.           │
└─────────────────────────────────────────────┘
```

**Key insight:** The agent doesn't just answer — it enters a **reasoning loop** where it can call tools, get results, reason again, and repeat until done.

---

## Multi-Turn Conversations

Agents can maintain conversation context across turns using `to_input_list()`:

In [ ]:
from agents import Agent, Runner

agent = Agent(
    name="Tutor",
    instructions="You are a Python tutor. Be concise. Remember context from previous messages."
)

# Turn 1
result = await Runner.run(agent, "What is a list in Python?")
print(f"Turn 1: {result.final_output}\n")

# Turn 2 — carry conversation history forward
new_input = result.to_input_list() + [{"role": "user", "content": "How do I sort one?"}]
result = await Runner.run(agent, new_input)
print(f"Turn 2: {result.final_output}\n")

# Turn 3
new_input = result.to_input_list() + [{"role": "user", "content": "What about reverse sort?"}]
result = await Runner.run(agent, new_input)
print(f"Turn 3: {result.final_output}")

### Key Pattern:
```python
# This is how you maintain conversation memory:
new_input = previous_result.to_input_list() + [{"role": "user", "content": "next message"}]
result = await Runner.run(agent, new_input)
```

---

## 3 Ways to Run an Agent

| Method | Use Case | Async? |
|---|---|---|
| `Runner.run()` | Jupyter, async apps, FastAPI | Yes |
| `Runner.run_sync()` | Scripts, CLI tools | No |
| `Runner.run_streamed()` | Real-time UIs, streaming | Yes |

In [ ]:
from agents import Agent, Runner

agent = Agent(name="Streamer", instructions="Explain briefly.")

# Streaming — see tokens as they arrive
result = Runner.run_streamed(agent, "What is machine learning in 2 sentences?")

async for event in result.stream_events():
    if event.type == "raw_response_event" and hasattr(event.data, 'delta'):
        print(event.data.delta, end="", flush=True)

print(f"\n\n--- Final: {result.final_output}")

---

## Exercise: Build Your First Agent

Create an agent that acts as a **Lahore travel guide**. Ask it 3 questions in a multi-turn conversation.

In [ ]:
# YOUR CODE HERE
# Hint:
# 1. Create Agent with name="Lahore Guide" and good instructions
# 2. Run it with a question about Lahore
# 3. Use to_input_list() to continue the conversation



---

## Summary — What You Learned

| Concept | Code |
|---|---|
| Create an agent | `Agent(name, instructions)` |
| Run async | `await Runner.run(agent, input)` |
| Run sync | `Runner.run_sync(agent, input)` |
| Run streaming | `Runner.run_streamed(agent, input)` |
| Get output | `result.final_output` |
| Continue conversation | `result.to_input_list() + [new_msg]` |

**Next:** Notebook 2 — Creating Proper Agents with instructions, output types, and model settings.